# Export route planner
For a product leaving a Saudi production site for a world market, lists every route: land leg (truck or crude pipeline) → export port that handles that cargo → sea route, with cost per tonne, days and a risk score. Writes `export_planner.html`, the interactive page for businesses.

**Risk** combines strait traffic disruption (IMF PortWatch, last 28 days against 2023), the Geopolitical Risk Index (Caldara and Iacoviello) and border crossings. **Today** closes straits whose traffic is down 90% or more; **normal** assumes no disruption.

Settings: `config/exports/` (products, sites, markets, ports, pipelines, costs; placeholders marked `dummy`) and `config/border_crossings.csv`.

In [ ]:
!pip -q install osmnx searoute osmium xlrd
!git clone -q --depth 1 https://github.com/danaalsh/jeddah-port-logistics.git repo 2>/dev/null || git -C repo pull -q

## 0. Settings

In [ ]:
import sys
from pathlib import Path

REPO = Path("repo")
sys.path.insert(0, str(REPO))
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/jeddah_port")
except ImportError:  # running outside Colab
    OUT_DIR = Path("out")
DATA_DIR, CACHE_DIR = OUT_DIR / "data", OUT_DIR / "cache"
for d in (OUT_DIR, DATA_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)
REFRESH_DATA = True  # download the latest weekly release files

## 1. Data, road network and risk signals

In [ ]:
import json
import pandas as pd
from sim import data, network, model, exports

files = data.fetch(["gcc_major.osm.pbf", "osm_date.txt", "portwatch_ports_list.csv",
                    "portwatch_chokepoints_daily.csv", "gpr_monthly.xls"], DATA_DIR,
                   refresh=REFRESH_DATA, optional=["gpr_monthly.xls"])
cfg = exports.load_config(REPO / "config/exports")
G = network.build_graph(files["gcc_major.osm.pbf"], CACHE_DIR, truck_max_kph=80)  # ~5 min first time
crossings = pd.read_csv(REPO / "config/border_crossings.csv")
network.add_border_delays(G, crossings)
network.set_costs(G, model.load_params(REPO / "config/costs.csv"))  # route choice weights for trucks

dis, choke_date = exports.disruption(pd.read_csv(files["portwatch_chokepoints_daily.csv"]))
gpr = exports.gpr_levels(files.get("gpr_monthly.xls"))
closures = exports.conditions(dis, cfg["p"])
print("closed today:", closures["today"], "| straits data to", choke_date)
print("GPR:", gpr or "not loaded yet")
pd.DataFrame(dis).T[["label", "base", "recent", "disruption"]]

## 2. Sites, ports and legs

In [ ]:
modon = pd.read_csv(REPO / "config/industrial_cities_modon.csv")
cfg["sites"] = pd.concat([cfg["sites"], pd.DataFrame(dict(
    site_id=modon.dest_id, name=modon.name, type="industrial_city", lat=modon.lat, lon=modon.lon,
    commodities="manufactured", source=modon.source))], ignore_index=True)
pw = pd.read_csv(files["portwatch_ports_list.csv"]).set_index("portid")[["lat", "lon"]]
cfg["ports"] = cfg["ports"].join(pw, on="portwatch_id")

land = exports.land_legs(G, cfg["sites"], cfg["ports"], cfg["pipelines"], crossings)   # ~2 min
seas = exports.sea_legs(cfg["ports"], cfg["markets"], closures)
print(f"{len(cfg['sites'])} sites, {len(land)} land legs, {len(seas)} sea legs")

## 3. Route options

In [ ]:
opts = exports.options(cfg, land, seas, dis, gpr, closures)
print(len(opts), "options;", opts.status.value_counts().to_dict())

# Example: crude oil from Abqaiq to China, today and normally
cols = ["condition", "port_key", "land_mode", "land_km", "sea_nm", "total_usd", "war_risk_usd", "days", "risk", "status", "reasons"]
opts[(opts.site_id == "ABQ") & (opts.market_id == "CHN")].sort_values(["condition", "total_usd"])[cols]

In [ ]:
# Cheapest open route today vs normally, for every export hub, product and market
best = (opts[opts.status == "open"].sort_values("total_usd")
        .groupby(["site_id", "commodity_id", "market_id", "condition"]).first().reset_index())
cmp = best.pivot_table(index=["site_id", "commodity_id", "market_id"], columns="condition",
                       values=["port_key", "total_usd", "risk"], aggfunc="first")
cmp[cmp.index.get_level_values("site_id").isin(cfg["sites"].query("type == 'export_hub'").site_id)]

## 4. Planner page

In [ ]:
meta = dict(choke_date=choke_date, osm_date=files["osm_date.txt"].read_text().strip())
page = exports.page_data(cfg, land, seas, dis, gpr, closures, opts, meta,
                         json.loads((REPO / "simulator/countries.geojson").read_text()))
size = exports.build_page(REPO / "simulator/planner_template.html", OUT_DIR / "export_planner.html", page)
opts.to_csv(OUT_DIR / "export_options.csv", index=False)
print(f"export_planner.html written ({size / 1e6:.1f} MB); open it in a browser")